<a href="https://colab.research.google.com/github/win-eva/als-sex-stratified-target-discovery/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pydeseq2

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from pathlib import Path

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

In [ ]:
drive.mount("/content/drive")
path = "/content/drive/MyDrive/ALS Data/AnswerALS_raw_counts_annotated.csv"
counts = pd.read_csv(path)

## Collapse to one sample per donor

In [ ]:
annotation_cols = ["Geneid", "symbol", "biotype", "description"]
sample_cols = [c for c in counts.columns if c not in annotation_cols]

sample_meta = pd.DataFrame({"sample_col": sample_cols})
sample_meta["donor_id"] = sample_meta["sample_col"].str.split("-", expand=True)[1]
sample_meta["libsize"] = counts[sample_cols].sum(axis=0).values

# repeat samples from the same donor correlate at r > 0.95, so any of them is a
# reasonable representative. Keeping the one with the deepest sequencing depth
# per donor
chosen = (
    sample_meta.sort_values("libsize", ascending=False)
    .drop_duplicates("donor_id")
)
selected_samples = chosen["sample_col"].tolist()

print("Selected donors:", chosen["donor_id"].nunique())
print("Selected samples:", len(selected_samples))

In [ ]:
counts_donor = pd.concat(
    [counts[annotation_cols], counts[selected_samples].copy()],
    axis=1
)
assert counts_donor[selected_samples].columns.is_unique

## Removing empty gene columns

In [ ]:
donor_cols = [c for c in counts_donor.columns if c not in annotation_cols]

# genes with at least one read in at least one donor
non_empty = counts_donor[donor_cols].sum(axis=1) > 0
counts_donor_nonempty = counts_donor.loc[non_empty].copy()

print("Empty genes removed:", (~non_empty).sum())
print("Remaining genes:", counts_donor_nonempty.shape[0])

In [ ]:
output_path = "/content/drive/MyDrive/ALS Data/ALS_counts_pre_normalisation_all_donors_collapsed.csv"
counts_donor_nonempty.to_csv(output_path, index=False)

## Normalisation (log2-CPM)

DESeq2 handles normalisation for the actual differential expression testing below,
but a simple log2-CPM version is useful for everything else (sex classification,
sanity-checking individual genes) where a full DESeq2 model per question is excessive.

In [ ]:
library_sizes = counts_donor_nonempty[donor_cols].sum(axis=0)
cpm = counts_donor_nonempty[donor_cols].div(library_sizes, axis=1) * 1e6
logcpm = np.log2(cpm + 1)

logcpm_df = pd.concat(
    [counts_donor_nonempty[annotation_cols].reset_index(drop=True),
     logcpm.reset_index(drop=True)],
    axis=1
)

logcpm_df.to_csv("/content/drive/MyDrive/ALS Data/ALS_logCPM_per_donor_all.csv", index=False)

## Sex classification and cohort split

No sex metadata came with the AnswerALS dataset, so sex is inferred from expression:
XIST is only transcribed from the inactive X chromosome and is essentially silent in
male cells, while the panel of Y-linked genes is essentially silent in female cells.
Comparing the two per donor gives a clean binary split without needing external
annotation.

In [ ]:
female_gene = "XIST"
male_genes = ["RPS4Y1", "DDX3Y", "KDM5D", "UTY", "USP9Y"]

sex_expr = logcpm_df.set_index("symbol")[donor_cols]

male_score = sex_expr.loc[sex_expr.index.isin(male_genes)].mean(axis=0)
female_score = sex_expr.loc[female_gene]

donor_sex = pd.DataFrame({
    "sample_id": donor_cols,
    "male_score": male_score.values,
    "female_score": female_score.values
})
donor_sex["sex"] = np.where(donor_sex["female_score"] > donor_sex["male_score"], "F", "M")
donor_sex["diagnosis"] = donor_sex["sample_id"].str.extract(r"^(CASE|CTRL)")[0]

donor_sex["sex"].value_counts()

In [ ]:
# Table 4: donor distribution after collapsing and sex classification
pd.crosstab(donor_sex["sex"], donor_sex["diagnosis"])

In [ ]:
male_samples = donor_sex.loc[donor_sex["sex"] == "M", "sample_id"].tolist()
female_samples = donor_sex.loc[donor_sex["sex"] == "F", "sample_id"].tolist()

male_counts = counts_donor_nonempty[annotation_cols + male_samples].copy()
female_counts = counts_donor_nonempty[annotation_cols + female_samples].copy()
male_logcpm = logcpm_df[annotation_cols + male_samples].copy()
female_logcpm = logcpm_df[annotation_cols + female_samples].copy()

male_counts.to_csv("/content/drive/MyDrive/ALS Data/ALS_raw_counts_male.csv", index=False)
female_counts.to_csv("/content/drive/MyDrive/ALS Data/ALS_raw_counts_female.csv", index=False)
male_logcpm.to_csv("/content/drive/MyDrive/ALS Data/ALS_logCPM_male.csv", index=False)
female_logcpm.to_csv("/content/drive/MyDrive/ALS Data/ALS_logCPM_female.csv", index=False)

## Differential expression with PyDESeq2

In [ ]:
ANNOTATION_COLS = ["Geneid", "symbol", "biotype", "description"]

def run_deseq2(csv_path, cohort_name, min_total_counts=50, min_nonzero_samples=5, n_cpus=2):
    df = pd.read_csv(csv_path)
    sample_cols = [c for c in df.columns if c not in ANNOTATION_COLS]

    count_df = df[["Geneid"] + sample_cols].copy()
    annotations = df[ANNOTATION_COLS].drop_duplicates(subset="Geneid").set_index("Geneid")
    del df
    gc.collect()

    count_df[sample_cols] = count_df[sample_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    count_df = count_df.groupby("Geneid", as_index=False)[sample_cols].sum()

    # light prefiltering before handing off to DESeq2's own independent filtering
    total_counts = count_df[sample_cols].sum(axis=1)
    nonzero_samples = (count_df[sample_cols] > 0).sum(axis=1)
    keep = (total_counts >= min_total_counts) & (nonzero_samples >= min_nonzero_samples)
    count_df = count_df.loc[keep].set_index("Geneid").round().astype(np.int32)

    metadata = pd.DataFrame(index=sample_cols)
    metadata["diagnosis"] = np.where(metadata.index.str.startswith("CASE"), "CASE", "CTRL")
    metadata["diagnosis"] = pd.Categorical(metadata["diagnosis"], categories=["CTRL", "CASE"])

    counts_for_deseq = count_df.T.copy()
    del count_df
    gc.collect()

    inference = DefaultInference(n_cpus=n_cpus)
    dds = DeseqDataSet(
        counts=counts_for_deseq,
        metadata=metadata,
        design="~ diagnosis",
        refit_cooks=False,
        low_memory=True,
        inference=inference,
    )
    del counts_for_deseq
    gc.collect()

    dds.deseq2()
    stats = DeseqStats(dds, contrast=["diagnosis", "CASE", "CTRL"], inference=inference)
    stats.summary()

    results_df = stats.results_df.reset_index().rename(columns={"index": "Geneid"})
    results_df = results_df.merge(annotations.reset_index(), on="Geneid", how="left")

    print(f"{cohort_name}: {dds.n_obs} donors, {dds.n_vars} genes after filtering")
    return results_df

In [ ]:
# ~12 min run time
female_results = run_deseq2("/content/drive/MyDrive/ALS Data/ALS_raw_counts_female.csv", "Female")
male_results = run_deseq2("/content/drive/MyDrive/ALS Data/ALS_raw_counts_male.csv", "Male")

female_results.to_csv("/content/drive/MyDrive/ALS Data/ALS_DESeq2_results_female.csv", index=False)
male_results.to_csv("/content/drive/MyDrive/ALS Data/ALS_DESeq2_results_male.csv", index=False)

## Quality filtering and volcano plots

Genes are kept if they're protein-coding (CLUE only accepts protein-coding query
genes further down the pipeline), pass the significance threshold, and clear a
minimum expression floor so the fold change estimate is trustworthy.

In [ ]:
def quality_filter(df):
    out = df.dropna(subset=["padj", "log2FoldChange"])
    return out[
        (out["biotype"] == "protein_coding") &
        (out["padj"] < 0.05) &
        (out["baseMean"] > 5)
    ].copy()

female_filt = quality_filter(female_results)
male_filt = quality_filter(male_results)

for name, df in [("Female", female_filt), ("Male", male_filt)]:
    n_up = (df["log2FoldChange"] > 0).sum()
    n_down = (df["log2FoldChange"] < 0).sum()
    print(f"{name}: {len(df)} DEGs ({n_up} up, {n_down} down)")

In [ ]:
def volcano_plot(results_df, title, n_samples):
    df = results_df.dropna(subset=["padj", "log2FoldChange"]).copy()
    df["neglog10padj"] = -np.log10(df["padj"])

    sig_up = (df["biotype"] == "protein_coding") & (df["padj"] < 0.05) & (df["baseMean"] > 5) & (df["log2FoldChange"] > 0)
    sig_down = (df["biotype"] == "protein_coding") & (df["padj"] < 0.05) & (df["baseMean"] > 5) & (df["log2FoldChange"] < 0)
    background = ~(sig_up | sig_down)

    plt.figure(figsize=(7, 6))
    plt.scatter(df.loc[background, "log2FoldChange"], df.loc[background, "neglog10padj"], s=5, alpha=0.25, color="grey")
    plt.scatter(df.loc[sig_up, "log2FoldChange"], df.loc[sig_up, "neglog10padj"], s=6, color="red", label="Upregulated")
    plt.scatter(df.loc[sig_down, "log2FoldChange"], df.loc[sig_down, "neglog10padj"], s=6, color="blue", label="Downregulated")

    plt.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.8)
    plt.text(df["log2FoldChange"].min(), -np.log10(0.05) + 0.5, f"{sig_up.sum()} up / {sig_down.sum()} down", fontsize=9)

    plt.xlabel("log2 fold change (ALS vs Control)")
    plt.ylabel("-log10(adjusted p-value)")
    plt.title(f"{title} (n = {n_samples})")
    plt.legend()
    plt.tight_layout()
    plt.show()

volcano_plot(female_results, "Female cohort", 475)
volcano_plot(male_results, "Male cohort", 628)

## CLUE query signature

In [ ]:
def prepare_for_clue(df):
    df = df.dropna(subset=["symbol", "log2FoldChange", "padj", "baseMean", "biotype"]).copy()
    df["symbol"] = df["symbol"].astype(str).str.strip()
    df = df[
        (df["symbol"] != "") &
        (df["padj"] < 0.05) &
        (df["baseMean"] > 5) &
        (df["biotype"] == "protein_coding")
    ]
    # a handful of Ensembl IDs map to the same symbol -> keep whichever has the
    # stronger effect so CLUE doesn't see the same gene twice
    df = df.sort_values("log2FoldChange", key=lambda x: x.abs(), ascending=False)
    return df.drop_duplicates(subset="symbol", keep="first")

female_ranked = prepare_for_clue(female_results)
male_ranked = prepare_for_clue(male_results)

In [ ]:
# Genes with invalid HGNC symbols or absent from the CLUE L1000 landmark set,
# checked manually against the CLUE platform (see Appendix, Table A1)
exclusions = {
    ("female", "up"): {
        "invalid": ["H3C8", "H3C2"],
        "valid_not_used": ["HOXA9", "HOXC9", "USP6", "LCN9", "OTP", "IRS4", "ZPLD1", "PRSS33", "SLC24A4", "CILP2", "RTL1", "OLIG1", "WNT7B", "WDR72", "PROKR1", "KLK4", "SHISA8"]
    },
    ("female", "down"): {
        "invalid": ["H1-1", "ANKRD30BL", "H2BC3", "QNG1", "H3-7", "C5orf63", "H4C6", "CCDC198", "CFAP107", "C3orf70", "C17orf100", "H4C16", "H2AJ"],
        "valid_not_used": ["ZNF578", "ZNF662", "POTEE", "FAM24B", "WBP2NL", "ZNF572", "POTEJ", "POTEI", "ZNF229", "PM20D1", "FAM135B", "ZNF560", "ZNF736", "ZFP3", "CCDC152", "NHLRC1", "ZNF829", "SLITRK4", "NKX1-2", "KCNK9", "NSMCE1", "TMC3", "LRRIQ3", "ARID3C", "ZNF717", "PGAM2", "SERF1A", "CCDC169", "LYPD5", "MAL2", "GDF6", "HPDL", "VWC2L", "ZNF454", "ZNF283", "ZNF300", "ONECUT3", "RASGRF2", "CEBPZOS", "SDSL", "ZNF585B", "ZNF311", "LRRC4C", "SPTSSB", "AP1S3", "ALG10B", "KCTD19", "HYLS1", "PCDHA13", "ESYT3", "GPR139", "ZNF502", "IQCD", "TMEM229B", "PLD6"]
    },
    ("male", "up"): {
        "invalid": ["C19orf18", "FLACC1", "C3orf52", "GARIN1A", "GARIN2", "C1orf116"],
        "valid_not_used": ["FAM83B", "OVCH2", "OVCH1", "ZPLD1", "TMPRSS9", "FOXD4L6", "ABHD12B", "A3GALT2", "PARVG", "BTBD16", "NPFF", "TMC3", "PNLDC1", "SLX1B", "MGAM2", "CLEC18A", "ANKRD30B", "OR56A1", "CBLN2", "FOXD4L4", "FOXI3", "CLEC18B", "SPOCD1", "RASEF", "FERMT3", "SLFNL1", "ZC3H12D", "SYCE1", "ASPDH", "MPP4", "RNF225", "CYP4Z1", "UROC1", "CCDC154", "ERAS", "ZNF626"]
    },
    ("male", "down"): {
        "invalid": ["H1-1", "H2BC3", "QNG1", "H3-7", "H4C6", "PLD5P1", "C5orf63", "TMEM273", "CIMIP5", "H2AJ"],
        "valid_not_used": ["ZNF578", "POTEE", "ZNF662", "FAM24B", "SLITRK2", "POTEI", "FAM135B", "ZNF572", "ZNF736", "PTPN20", "TMEM132C", "PGAM2", "PM20D1", "SLITRK4", "ZFP3", "COL22A1", "COL6A6", "POTEF", "ZNF829", "NHLRC1", "TUBB8", "ZNF560", "ABCG8", "FAM110C", "BHLHE22", "RSPO1", "ZNF229", "CCDC152", "ZNF534", "IRX2", "KCNK9", "UNC5D", "ZNF717", "FBXO32", "NSMCE1", "IRX1", "SBK2", "FRMPD2", "CCBE1", "TMEM163", "SLC25A48", "TC2N", "TMC2", "PCP4L1"]
    }
}

In [ ]:
outdir = Path("/content/drive/MyDrive/ALS Data")

def build_clue_signature(df, sex, top_n=50):
    audit = []

    for direction in ["up", "down"]:
        exclude = set(exclusions[(sex, direction)]["invalid"]) | set(exclusions[(sex, direction)]["valid_not_used"])

        if direction == "up":
            ranked = df[df["log2FoldChange"] > 0].sort_values("log2FoldChange", ascending=False)
        else:
            ranked = df[df["log2FoldChange"] < 0].sort_values("log2FoldChange", ascending=True)

        kept = ranked[~ranked["symbol"].isin(exclude)].head(top_n)

        kept["symbol"].to_csv(outdir / f"{sex}_CLUE_{direction}_top{top_n}_refilled.txt", index=False, header=False)
        kept[["Geneid", "symbol", "log2FoldChange", "padj", "baseMean"]].to_csv(
            outdir / f"{sex}_CLUE_{direction}_top{top_n}_refilled_mapping.csv", index=False
        )

        print(f"{sex} {direction}: {len(kept)} genes saved ({len(ranked[ranked['symbol'].isin(exclude)])} excluded and refilled)")
        audit.append((direction, exclude, kept))

    return audit

female_audit = build_clue_signature(female_ranked, "female")
male_audit = build_clue_signature(male_ranked, "male")

In [ ]:
# audit trail for the appendix
audit_rows = []
for sex, results in [("female", female_audit), ("male", male_audit)]:
    for direction, excluded_genes, _ in results:
        for gene in exclusions[(sex, direction)]["invalid"]:
            audit_rows.append({"sex": sex, "direction": direction, "gene": gene, "reason": "invalid gene symbol"})
        for gene in exclusions[(sex, direction)]["valid_not_used"]:
            audit_rows.append({"sex": sex, "direction": direction, "gene": gene, "reason": "valid but absent from CLUE L1000 landmark set"})

pd.DataFrame(audit_rows).to_csv(outdir / "CLUE_manual_exclusion_record.csv", index=False)